# Maya-CSM TTS Server (Colab)

Runs the Maya TTS API on a free Colab GPU and exposes it via a Cloudflare tunnel for SillyTavern.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**.
2. Accept the model terms at https://huggingface.co/sesame/csm-1b (once, with your HF account).
3. Add a Colab secret named `HF_TOKEN` (key icon in the left sidebar) with a Hugging Face read token, and enable notebook access.
4. Set `REPO_URL` below to your fork/copy of the maya-csm repo (push this project to GitHub first).

In [ ]:
REPO_URL = "https://github.com/l0ophole/maya-csm"  # <-- change me

!git clone -q $REPO_URL /content/maya-csm
%pip install -q -e /content/maya-csm[model]
# Colab preinstalls may predate CSM support in transformers
%pip install -q -U transformers peft accelerate

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["MAYA_ADAPTER"] = "shb777/csm-maya-exp2"  # pull LoRA from the Hub
os.environ["MAYA_PRELOAD"] = "1"

In [ ]:
# Start the server in a background thread (model loads now; takes a few minutes first run)
import threading
import uvicorn
from maya_csm.config import Settings
from maya_csm.server import create_app

app = create_app(Settings.from_env())
threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"),
    daemon=True,
).start()
print("server starting on :8000")

In [ ]:
# Cloudflare tunnel (no account needed). Re-run this cell if the URL dies.
import re
import subprocess
import time

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
url = None
deadline = time.time() + 30
while time.time() < deadline and url is None:
    line = proc.stdout.readline()
    m = re.search(r"https://[\w.-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
print("Public URL:", url)
print("\nSillyTavern OpenAI-Compatible TTS endpoint:")
print(f"  {url}/v1/audio/speech")
print("\nSmoke test:")
print(f"  curl -X POST {url}/v1/audio/speech -H 'Content-Type: application/json' "
      "-d '{\"input\":\"[laughing] That is hilarious.\",\"voice\":\"maya\"}' -o out.wav")

### Fallback: ngrok tunnel
If trycloudflare is throttled, add a Colab secret `NGROK_AUTH_TOKEN` (from https://dashboard.ngrok.com) and run:

In [ ]:
# %pip install -q pyngrok
# from pyngrok import ngrok
# ngrok.set_auth_token(userdata.get("NGROK_AUTH_TOKEN"))
# print("Public URL:", ngrok.connect(8000, "http").public_url)

In [ ]:
# In-notebook sanity check (no tunnel needed)
import requests
from IPython.display import Audio

r = requests.post(
    "http://localhost:8000/v1/audio/speech",
    json={"input": "[giggling] Hey there, it's so good to hear your voice.", "voice": "maya"},
)
r.raise_for_status()
Audio(r.content)